# 08 — LLM Response Validation and Inference Freeze

## Purpose

This notebook independently audits the formal `primary_46` LLM inference artifacts produced by Notebook 07 and freezes the validated inference dataset for downstream evaluation.

The audit verifies artifact hashes, request identity, paired coverage, execution-contract fields, JSON parsing, response-schema compliance, legal predicted labels, citation count and distinctness, supported feature names, exact observed-value grounding, and preserved timeout or error evidence.

This notebook does not load, join, inspect, or infer private ground truth. No model request is made, and no failed or invalid response is retried or repaired.

In [1]:
from __future__ import annotations

import hashlib
import json
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from jsonschema import Draft202012Validator


def find_project_root(start: Path) -> Path:
    """Locate the repository root from the project or notebooks folder."""
    start = start.resolve()

    for candidate in (start, *start.parents):
        if (
            (candidate / "README.md").is_file()
            and (candidate / "notebooks").is_dir()
            and (candidate / "configs").is_dir()
        ):
            return candidate

    raise FileNotFoundError(
        "Could not locate the project root containing README.md, "
        "notebooks/ and configs/."
    )


PROJECT_ROOT = find_project_root(Path.cwd())

REPRESENTATION_DIR = (
    PROJECT_ROOT
    / "data"
    / "interim"
    / "representations"
    / "primary_46"
)

STRUCTURED_PATH = (
    REPRESENTATION_DIR / "structured.jsonl"
)
TEXT_PATH = (
    REPRESENTATION_DIR / "deterministic_text.jsonl"
)
EQUIVALENCE_PATH = (
    REPRESENTATION_DIR / "equivalence_validation.csv"
)
REPRESENTATION_MANIFEST_PATH = (
    REPRESENTATION_DIR / "manifest.json"
)

PROTOCOL_PATH = (
    PROJECT_ROOT
    / "configs"
    / "llm_protocol_primary_46.json"
)
OUTPUT_SCHEMA_PATH = (
    PROJECT_ROOT
    / "configs"
    / "llm_output_schema.json"
)

BATCH_RESULTS_DIR = (
    PROJECT_ROOT
    / "results"
    / "inference"
    / "primary_46"
)
BATCH_RESULTS_JSONL_PATH = (
    BATCH_RESULTS_DIR
    / "opencode_batch_results.jsonl"
)
BATCH_MANIFEST_PATH = (
    BATCH_RESULTS_DIR
    / "opencode_batch_manifest.json"
)
BATCH_CHECKPOINT_DIR = (
    BATCH_RESULTS_DIR / "checkpoints"
)

VALIDATION_RESULTS_DIR = (
    PROJECT_ROOT
    / "results"
    / "validation"
    / "primary_46"
)
FROZEN_INFERENCE_PATH = (
    VALIDATION_RESULTS_DIR
    / "opencode_frozen_inference.jsonl"
)
VALIDATION_MANIFEST_PATH = (
    VALIDATION_RESULTS_DIR
    / "opencode_validation_manifest.json"
)


required_paths = [
    STRUCTURED_PATH,
    TEXT_PATH,
    EQUIVALENCE_PATH,
    REPRESENTATION_MANIFEST_PATH,
    PROTOCOL_PATH,
    OUTPUT_SCHEMA_PATH,
    BATCH_RESULTS_JSONL_PATH,
    BATCH_MANIFEST_PATH,
    BATCH_CHECKPOINT_DIR,
]

missing_paths = [
    str(path)
    for path in required_paths
    if not path.exists()
]

if missing_paths:
    raise FileNotFoundError(
        "Required Notebook 08 inputs are missing:\n- "
        + "\n- ".join(missing_paths)
    )


environment_summary = pd.Series(
    {
        "project_root": str(PROJECT_ROOT),
        "batch_results_jsonl": str(
            BATCH_RESULTS_JSONL_PATH
        ),
        "batch_manifest": str(
            BATCH_MANIFEST_PATH
        ),
        "batch_checkpoint_directory": str(
            BATCH_CHECKPOINT_DIR
        ),
        "validation_results_directory": str(
            VALIDATION_RESULTS_DIR
        ),
        "required_paths_checked": len(
            required_paths
        ),
        "missing_required_paths": len(
            missing_paths
        ),
        "ground_truth_path_defined": False,
        "ground_truth_loaded": False,
        "network_request_made": False,
        "validation_artifact_written": False,
    },
    name="value",
)

environment_summary

project_root                        /Users/ruiwang/Developer/compsci742-rui-pilot
batch_results_jsonl             /Users/ruiwang/Developer/compsci742-rui-pilot/...
batch_manifest                  /Users/ruiwang/Developer/compsci742-rui-pilot/...
batch_checkpoint_directory      /Users/ruiwang/Developer/compsci742-rui-pilot/...
validation_results_directory    /Users/ruiwang/Developer/compsci742-rui-pilot/...
required_paths_checked                                                          9
missing_required_paths                                                          0
ground_truth_path_defined                                                   False
ground_truth_loaded                                                         False
network_request_made                                                        False
validation_artifact_written                                                 False
Name: value, dtype: object

## 1. Independent artifact-integrity and checkpoint-lineage audit

This section independently reloads the formal inference artifacts from disk, recomputes all recorded SHA-256 digests, verifies the 400-file checkpoint inventory, and confirms that every aggregated result is an exact copy of its source checkpoint plus the declared ordering metadata.

No response is repaired or retried, and private ground truth remains inaccessible.

In [2]:
def read_jsonl(path: Path) -> list[dict]:
    """Read a UTF-8 JSONL file as a list of JSON objects."""
    records = []

    with path.open("r", encoding="utf-8") as handle:
        for line_number, raw_line in enumerate(
            handle,
            start=1,
        ):
            stripped_line = raw_line.strip()

            if not stripped_line:
                continue

            try:
                record = json.loads(stripped_line)
            except json.JSONDecodeError as exc:
                raise ValueError(
                    f"Invalid JSON in {path.name}, "
                    f"line {line_number}."
                ) from exc

            if not isinstance(record, dict):
                raise ValueError(
                    f"Line {line_number} in {path.name} "
                    "is not an object."
                )

            records.append(record)

    return records


def sha256_file(path: Path) -> str:
    """Calculate the SHA-256 digest of a file's exact bytes."""
    return hashlib.sha256(
        path.read_bytes()
    ).hexdigest()


def stable_json_sha256(value: object) -> str:
    """Calculate a deterministic digest for a JSON-compatible value."""
    canonical_json = json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
    )

    return hashlib.sha256(
        canonical_json.encode("utf-8")
    ).hexdigest()


def project_relative_path(path: Path) -> str:
    """Return a stable project-relative POSIX path."""
    return path.resolve().relative_to(
        PROJECT_ROOT.resolve()
    ).as_posix()


structured_records = read_jsonl(
    STRUCTURED_PATH
)
text_records = read_jsonl(
    TEXT_PATH
)
equivalence_validation = pd.read_csv(
    EQUIVALENCE_PATH
)
batch_results = read_jsonl(
    BATCH_RESULTS_JSONL_PATH
)

with REPRESENTATION_MANIFEST_PATH.open(
    "r",
    encoding="utf-8",
) as handle:
    representation_manifest = json.load(handle)

with PROTOCOL_PATH.open(
    "r",
    encoding="utf-8",
) as handle:
    llm_protocol = json.load(handle)

with OUTPUT_SCHEMA_PATH.open(
    "r",
    encoding="utf-8",
) as handle:
    llm_output_schema = json.load(handle)

with BATCH_MANIFEST_PATH.open(
    "r",
    encoding="utf-8",
) as handle:
    batch_manifest = json.load(handle)


assert isinstance(
    representation_manifest,
    dict,
)
assert isinstance(llm_protocol, dict)
assert isinstance(llm_output_schema, dict)
assert isinstance(batch_manifest, dict)

assert len(structured_records) == 200
assert len(text_records) == 200
assert len(equivalence_validation) == 200
assert len(batch_results) == 400

assert (
    batch_manifest["manifest_version"]
    == "0.1.0"
)
assert (
    batch_manifest["experiment_phase"]
    == "formal_batch_inference"
)
assert batch_manifest["feature_set"] == {
    "id": "primary_46",
    "feature_count": 46,
}
assert (
    batch_manifest["request_plan"][
        "paired_sample_count"
    ]
    == 200
)
assert (
    batch_manifest["request_plan"][
        "request_count"
    ]
    == 400
)
assert (
    batch_manifest["safety"][
        "ground_truth_path_defined"
    ]
    is False
)
assert (
    batch_manifest["safety"][
        "ground_truth_loaded"
    ]
    is False
)
assert (
    batch_manifest["safety"][
        "credentials_retained"
    ]
    is False
)
assert (
    batch_manifest["safety"][
        "hidden_reasoning_retained"
    ]
    is False
)


locked_paths = {
    "structured_jsonl": STRUCTURED_PATH,
    "deterministic_text_jsonl": TEXT_PATH,
    "equivalence_validation_csv": (
        EQUIVALENCE_PATH
    ),
    "representation_manifest": (
        REPRESENTATION_MANIFEST_PATH
    ),
    "llm_protocol": PROTOCOL_PATH,
    "llm_output_schema": (
        OUTPUT_SCHEMA_PATH
    ),
    "reliability_manifest": (
        PROJECT_ROOT
        / "results"
        / "reliability"
        / "primary_46"
        / "opencode_reliability_manifest.json"
    ),
}

locked_hash_rows = []

for artifact_name, artifact_path in (
    locked_paths.items()
):
    if not artifact_path.is_file():
        raise FileNotFoundError(
            f"Missing locked artifact: "
            f"{artifact_path}"
        )

    manifest_entry = batch_manifest[
        "locked_artifacts"
    ][artifact_name]

    observed_relative_path = (
        project_relative_path(
            artifact_path
        )
    )
    observed_sha256 = sha256_file(
        artifact_path
    )

    relative_path_matches = (
        manifest_entry["relative_path"]
        == observed_relative_path
    )
    sha256_matches = (
        manifest_entry["sha256"]
        == observed_sha256
    )

    assert relative_path_matches
    assert sha256_matches

    locked_hash_rows.append(
        {
            "artifact": artifact_name,
            "relative_path": (
                observed_relative_path
            ),
            "sha256": observed_sha256,
            "relative_path_matches": (
                relative_path_matches
            ),
            "sha256_matches": (
                sha256_matches
            ),
        }
    )


locked_hash_table = pd.DataFrame(
    locked_hash_rows
)

results_manifest_entry = (
    batch_manifest["results_artifact"]
)
results_jsonl_sha256 = sha256_file(
    BATCH_RESULTS_JSONL_PATH
)

assert (
    results_manifest_entry[
        "relative_path"
    ]
    == project_relative_path(
        BATCH_RESULTS_JSONL_PATH
    )
)
assert (
    results_manifest_entry[
        "record_count"
    ]
    == len(batch_results)
    == 400
)
assert (
    results_manifest_entry[
        "byte_count"
    ]
    == BATCH_RESULTS_JSONL_PATH.stat().st_size
)
assert (
    results_manifest_entry["sha256"]
    == results_jsonl_sha256
)
assert (
    results_manifest_entry["ordering"]
    == "request_plan_position_ascending"
)


checkpoint_inventory = batch_manifest[
    "checkpoint_inventory"
]

assert isinstance(
    checkpoint_inventory,
    list,
)
assert len(checkpoint_inventory) == 400
assert (
    batch_manifest[
        "checkpoint_inventory_sha256"
    ]
    == stable_json_sha256(
        checkpoint_inventory
    )
)

checkpoint_paths = sorted(
    BATCH_CHECKPOINT_DIR.glob("*.json")
)
assert len(checkpoint_paths) == 400

manifest_checkpoint_paths = {
    entry["relative_path"]
    for entry in checkpoint_inventory
}
observed_checkpoint_paths = {
    project_relative_path(path)
    for path in checkpoint_paths
}

assert (
    manifest_checkpoint_paths
    == observed_checkpoint_paths
)


AGGREGATE_ONLY_FIELDS = {
    "request_plan_position",
    "order_group",
    "within_pair_position",
    "checkpoint_relative_path",
    "checkpoint_file_sha256",
}

lineage_rows = []

for expected_position, (
    result_record,
    inventory_entry,
) in enumerate(
    zip(
        batch_results,
        checkpoint_inventory,
        strict=True,
    )
):
    assert (
        result_record[
            "request_plan_position"
        ]
        == expected_position
    )
    assert (
        inventory_entry[
            "request_plan_position"
        ]
        == expected_position
    )

    assert (
        result_record["record_index"]
        == inventory_entry["record_index"]
    )
    assert (
        result_record["sample_id"]
        == inventory_entry["sample_id"]
    )
    assert (
        result_record["condition"]
        == inventory_entry["condition"]
    )

    checkpoint_path = (
        PROJECT_ROOT
        / inventory_entry["relative_path"]
    ).resolve()

    checkpoint_path.relative_to(
        PROJECT_ROOT.resolve()
    )

    assert (
        checkpoint_path.parent
        == BATCH_CHECKPOINT_DIR.resolve()
    )
    assert checkpoint_path.is_file()

    checkpoint_sha256 = sha256_file(
        checkpoint_path
    )

    assert (
        checkpoint_sha256
        == inventory_entry["sha256"]
    )
    assert (
        checkpoint_sha256
        == result_record[
            "checkpoint_file_sha256"
        ]
    )
    assert (
        result_record[
            "checkpoint_relative_path"
        ]
        == inventory_entry["relative_path"]
    )

    with checkpoint_path.open(
        "r",
        encoding="utf-8",
    ) as handle:
        checkpoint_record = json.load(
            handle
        )

    result_checkpoint_payload = {
        field_name: field_value
        for field_name, field_value
        in result_record.items()
        if field_name
        not in AGGREGATE_ONLY_FIELDS
    }

    assert (
        result_checkpoint_payload
        == checkpoint_record
    )

    lineage_rows.append(
        {
            "request_plan_position": (
                expected_position
            ),
            "record_index": (
                result_record[
                    "record_index"
                ]
            ),
            "sample_id": (
                result_record[
                    "sample_id"
                ]
            ),
            "condition": (
                result_record[
                    "condition"
                ]
            ),
            "checkpoint_sha256_matches": True,
            "aggregate_payload_matches_checkpoint": True,
        }
    )


lineage_table = pd.DataFrame(
    lineage_rows
)

assert lineage_table[
    ["record_index", "condition"]
].drop_duplicates().shape[0] == 400

assert (
    lineage_table["sample_id"].nunique()
    == 200
)
assert lineage_table[
    "checkpoint_sha256_matches"
].all()
assert lineage_table[
    "aggregate_payload_matches_checkpoint"
].all()
assert set(
    lineage_table["condition"]
) == {
    "structured",
    "deterministic_text",
}
assert not any(
    record["ground_truth_loaded"]
    for record in batch_results
)


artifact_integrity_summary = pd.Series(
    {
        "locked_artifact_count": len(
            locked_hash_table
        ),
        "locked_artifact_hashes_matched": int(
            locked_hash_table[
                "sha256_matches"
            ].sum()
        ),
        "batch_results_record_count": len(
            batch_results
        ),
        "batch_results_sha256": (
            results_jsonl_sha256
        ),
        "batch_manifest_sha256": (
            sha256_file(
                BATCH_MANIFEST_PATH
            )
        ),
        "checkpoint_file_count": len(
            checkpoint_paths
        ),
        "checkpoint_inventory_hash_matched": True,
        "checkpoint_hashes_matched": int(
            lineage_table[
                "checkpoint_sha256_matches"
            ].sum()
        ),
        "aggregate_payloads_matched_checkpoints": int(
            lineage_table[
                "aggregate_payload_matches_checkpoint"
            ].sum()
        ),
        "ordered_request_positions_verified": True,
        "unique_request_keys": (
            lineage_table[
                ["record_index", "condition"]
            ]
            .drop_duplicates()
            .shape[0]
        ),
        "paired_sample_count": (
            lineage_table[
                "sample_id"
            ].nunique()
        ),
        "ground_truth_loaded": False,
        "network_request_made": False,
        "validation_artifact_written": False,
    },
    name="value",
)

display(locked_hash_table)

artifact_integrity_summary

,artifact,relative_path,sha256,relative_path_matches,sha256_matches
0,structured_jsonl,data/interim/representations/primary_46/struct...,a74fdbb25e8d24fb9afd06d13982f19689a04685436a65...,True,True
1,deterministic_text_jsonl,data/interim/representations/primary_46/determ...,e47549bc5c5ae08d170e46c43b223afdaae6d3aaef7106...,True,True
2,equivalence_validation_csv,data/interim/representations/primary_46/equiva...,a3c036f068e2dcc2a502b25ac34c390daa6c20f0c0bcc3...,True,True
3,representation_manifest,data/interim/representations/primary_46/manife...,c653d611be838dc7dfb98b04ab22a390e61a38eed1c5dd...,True,True
4,llm_protocol,configs/llm_protocol_primary_46.json,3ff01d1b7fa91f0b38d03a4324034e26e71c3288b74cd3...,True,True
5,llm_output_schema,configs/llm_output_schema.json,1103a917ce1cad918ab4a94c5d94d8a0299400749ca9e0...,True,True
6,reliability_manifest,results/reliability/primary_46/opencode_reliab...,a3af39ed2123b22a8450bfd29b85de984d27ec8a16cd88...,True,True


locked_artifact_count                                                                     7
locked_artifact_hashes_matched                                                            7
batch_results_record_count                                                              400
batch_results_sha256                      fdca52ad78ba136794e7a4a9d5bc684788edcdae180f96...
batch_manifest_sha256                     994e6a1b69c88ea78475aafe7224af4a53916d8362e004...
checkpoint_file_count                                                                   400
checkpoint_inventory_hash_matched                                                      True
checkpoint_hashes_matched                                                               400
aggregate_payloads_matched_checkpoints                                                  400
ordered_request_positions_verified                                                     True
unique_request_keys                                                             

## 2. Independent request-contract reconstruction

This section independently reconstructs every locked prompt and OpenCode execution contract from the original paired representations and protocol files.

It verifies request identity, canonical-payload hashes, alternating condition order, prompt hashes, execution-contract hashes, backend settings, safety controls, and the absence of sample identifiers from model prompts.

In [3]:
LOCKED_OPENCODE_EXECUTABLE = Path(
    "/opt/homebrew/bin/opencode"
)
LOCKED_BACKEND = "opencode"
LOCKED_OPENCODE_VERSION = "1.18.22"
LOCKED_MODEL_ROUTE = "uoa/MiniMax-M3"
LOCKED_PROMPT_TEMPLATE_VERSION = "0.1.0"
LOCKED_TIMEOUT_SECONDS = 300


assert (
    batch_manifest["backend"]["name"]
    == LOCKED_BACKEND
)
assert (
    batch_manifest["backend"]["version"]
    == LOCKED_OPENCODE_VERSION
)
assert (
    batch_manifest["backend"]["model_route"]
    == LOCKED_MODEL_ROUTE
)
assert (
    batch_manifest["backend"][
        "prompt_template_version"
    ]
    == LOCKED_PROMPT_TEMPLATE_VERSION
)
assert (
    batch_manifest["backend"][
        "subprocess_timeout_seconds"
    ]
    == LOCKED_TIMEOUT_SECONDS
)
assert (
    batch_manifest["backend"][
        "maximum_concurrent_requests"
    ]
    == 3
)
assert (
    batch_manifest["backend"][
        "automatic_retry_used"
    ]
    is False
)


COMMON_SYSTEM_INSTRUCTION = (
    llm_protocol["instructions"][
        "system_instruction"
    ]
)
RECORD_OPENING_DELIMITER = (
    llm_protocol["instructions"][
        "record_opening_delimiter"
    ]
)
RECORD_CLOSING_DELIMITER = (
    llm_protocol["instructions"][
        "record_closing_delimiter"
    ]
)

OPENCODE_RESEARCH_SCHEMA_TEXT = json.dumps(
    llm_output_schema["schema"],
    ensure_ascii=False,
    sort_keys=True,
    separators=(",", ":"),
)


def build_user_message(
    model_input: str,
) -> str:
    """Rebuild the locked record-delimited user message."""
    if (
        not isinstance(model_input, str)
        or not model_input.strip()
    ):
        raise ValueError(
            "model_input must be a non-empty string."
        )

    if (
        RECORD_OPENING_DELIMITER
        in model_input
        or RECORD_CLOSING_DELIMITER
        in model_input
    ):
        raise ValueError(
            "model_input contains a reserved delimiter."
        )

    return (
        f"{RECORD_OPENING_DELIMITER}\n"
        f"{model_input}\n"
        f"{RECORD_CLOSING_DELIMITER}"
    )


def build_logical_request_contract(
    model_input: str,
) -> dict:
    """Rebuild the provider-independent logical request."""
    return {
        "system_instruction": (
            COMMON_SYSTEM_INSTRUCTION
        ),
        "user_message": build_user_message(
            model_input
        ),
        "output_schema": llm_output_schema,
    }


def build_opencode_research_prompt(
    request_contract: dict,
) -> str:
    """Rebuild prompt-template version 0.1.0 exactly."""
    required_fields = {
        "system_instruction",
        "user_message",
        "output_schema",
    }

    if set(request_contract) != required_fields:
        raise ValueError(
            "Unexpected logical request fields."
        )

    if (
        request_contract["output_schema"]
        != llm_output_schema
    ):
        raise ValueError(
            "Request does not use the locked schema."
        )

    return (
        "<research_instruction>\n"
        f"{request_contract['system_instruction']}\n"
        "</research_instruction>\n\n"
        "<required_output_json_schema>\n"
        f"{OPENCODE_RESEARCH_SCHEMA_TEXT}\n"
        "</required_output_json_schema>\n\n"
        "<research_input>\n"
        f"{request_contract['user_message']}\n"
        "</research_input>\n\n"
        "Follow the research instruction using only the "
        "enclosed research input. Return only the JSON "
        "object required by the enclosed schema."
    )


def calculate_prompt_sha256(
    prompt: str,
) -> str:
    """Reproduce the locked prompt-digest construction."""
    return stable_json_sha256(
        {
            "template_version": (
                LOCKED_PROMPT_TEMPLATE_VERSION
            ),
            "prompt": prompt,
        }
    )


def build_execution_contract(
    prompt_sha256: str,
) -> dict:
    """Rebuild the exact OpenCode execution contract."""
    return {
        "backend": LOCKED_BACKEND,
        "opencode_version": (
            LOCKED_OPENCODE_VERSION
        ),
        "executable": str(
            LOCKED_OPENCODE_EXECUTABLE
        ),
        "model": LOCKED_MODEL_ROUTE,
        "prompt_template_version": (
            LOCKED_PROMPT_TEMPLATE_VERSION
        ),
        "prompt_sha256": prompt_sha256,
        "pure": True,
        "empty_temporary_directory": True,
        "files_attached": False,
        "auto_approval": False,
        "format": "json",
        "timeout_seconds": (
            LOCKED_TIMEOUT_SECONDS
        ),
        "maximum_model_output_tokens": None,
    }


structured_sample_ids = [
    record["sample_id"]
    for record in structured_records
]
text_sample_ids = [
    record["sample_id"]
    for record in text_records
]

assert (
    structured_sample_ids
    == text_sample_ids
)
assert len(
    set(structured_sample_ids)
) == 200

for structured_record, text_record in zip(
    structured_records,
    text_records,
    strict=True,
):
    assert (
        structured_record["sample_id"]
        == text_record["sample_id"]
    )
    assert (
        structured_record["feature_set_id"]
        == "primary_46"
    )
    assert (
        text_record["feature_set_id"]
        == "primary_46"
    )
    assert (
        structured_record[
            "canonical_payload_sha256"
        ]
        == text_record[
            "canonical_payload_sha256"
        ]
    )
    assert (
        structured_record["sample_id"]
        not in structured_record["model_input"]
    )
    assert (
        text_record["sample_id"]
        not in text_record["model_input"]
    )


request_contract_rows = []

for record_index in range(200):
    structured_record = (
        structured_records[record_index]
    )
    text_record = (
        text_records[record_index]
    )

    if record_index % 2 == 0:
        expected_condition_order = [
            "structured",
            "deterministic_text",
        ]
        expected_order_group = (
            "structured_first"
        )
    else:
        expected_condition_order = [
            "deterministic_text",
            "structured",
        ]
        expected_order_group = (
            "deterministic_text_first"
        )

    source_by_condition = {
        "structured": structured_record,
        "deterministic_text": text_record,
    }

    pair_results = batch_results[
        record_index * 2:
        record_index * 2 + 2
    ]

    assert [
        result["condition"]
        for result in pair_results
    ] == expected_condition_order

    for (
        within_pair_position,
        result_record,
    ) in enumerate(
        pair_results,
        start=1,
    ):
        condition = (
            result_record["condition"]
        )
        source_record = (
            source_by_condition[condition]
        )

        assert (
            result_record[
                "request_plan_position"
            ]
            == (
                record_index * 2
                + within_pair_position
                - 1
            )
        )
        assert (
            result_record["record_index"]
            == record_index
        )
        assert (
            result_record["sample_id"]
            == source_record["sample_id"]
        )
        assert (
            result_record["condition"]
            == condition
        )
        assert (
            result_record["order_group"]
            == expected_order_group
        )
        assert (
            result_record[
                "within_pair_position"
            ]
            == within_pair_position
        )
        assert (
            result_record[
                "canonical_payload_sha256"
            ]
            == source_record[
                "canonical_payload_sha256"
            ]
        )
        assert (
            result_record["feature_set_id"]
            == "primary_46"
        )

        logical_contract = (
            build_logical_request_contract(
                source_record["model_input"]
            )
        )
        rebuilt_prompt = (
            build_opencode_research_prompt(
                logical_contract
            )
        )
        rebuilt_prompt_sha256 = (
            calculate_prompt_sha256(
                rebuilt_prompt
            )
        )

        assert (
            result_record["sample_id"]
            not in rebuilt_prompt
        )
        assert (
            result_record["prompt_sha256"]
            == rebuilt_prompt_sha256
        )

        rebuilt_execution_contract_sha256 = (
            stable_json_sha256(
                build_execution_contract(
                    rebuilt_prompt_sha256
                )
            )
        )

        assert (
            result_record[
                "execution_contract_sha256"
            ]
            == (
                rebuilt_execution_contract_sha256
            )
        )

        assert (
            result_record["backend"]
            == LOCKED_BACKEND
        )
        assert (
            result_record["opencode_version"]
            == LOCKED_OPENCODE_VERSION
        )
        assert (
            result_record["requested_model"]
            == LOCKED_MODEL_ROUTE
        )
        assert (
            result_record[
                "prompt_template_version"
            ]
            == LOCKED_PROMPT_TEMPLATE_VERSION
        )
        assert (
            result_record[
                "subprocess_timeout_seconds"
            ]
            == LOCKED_TIMEOUT_SECONDS
        )
        assert (
            result_record["pure_mode_used"]
            is True
        )
        assert (
            result_record[
                "empty_temporary_directory_used"
            ]
            is True
        )
        assert (
            result_record["files_attached"]
            is False
        )
        assert (
            result_record[
                "auto_approval_used"
            ]
            is False
        )
        assert (
            result_record[
                "maximum_model_output_tokens_supplied"
            ]
            is False
        )
        assert (
            result_record[
                "sample_id_sent_to_model"
            ]
            is False
        )
        assert (
            result_record[
                "ground_truth_loaded"
            ]
            is False
        )

        request_contract_rows.append(
            {
                "request_plan_position": (
                    result_record[
                        "request_plan_position"
                    ]
                ),
                "record_index": (
                    record_index
                ),
                "sample_id": (
                    result_record[
                        "sample_id"
                    ]
                ),
                "condition": condition,
                "order_group": (
                    expected_order_group
                ),
                "within_pair_position": (
                    within_pair_position
                ),
                "canonical_payload_sha256_matches": True,
                "prompt_sha256_matches": True,
                "execution_contract_sha256_matches": True,
                "sample_id_absent_from_prompt": True,
                "safety_contract_matches": True,
            }
        )


request_contract_table = pd.DataFrame(
    request_contract_rows
)

assert len(request_contract_table) == 400
assert request_contract_table[
    ["record_index", "condition"]
].drop_duplicates().shape[0] == 400

assert request_contract_table[
    "canonical_payload_sha256_matches"
].all()
assert request_contract_table[
    "prompt_sha256_matches"
].all()
assert request_contract_table[
    "execution_contract_sha256_matches"
].all()
assert request_contract_table[
    "sample_id_absent_from_prompt"
].all()
assert request_contract_table[
    "safety_contract_matches"
].all()


request_contract_condition_summary = (
    request_contract_table
    .groupby(
        "condition",
        sort=True,
    )
    .agg(
        request_count=(
            "sample_id",
            "size",
        ),
        unique_sample_count=(
            "sample_id",
            "nunique",
        ),
        payload_hash_matches=(
            "canonical_payload_sha256_matches",
            "sum",
        ),
        prompt_hash_matches=(
            "prompt_sha256_matches",
            "sum",
        ),
        execution_contract_hash_matches=(
            "execution_contract_sha256_matches",
            "sum",
        ),
        safety_contract_matches=(
            "safety_contract_matches",
            "sum",
        ),
    )
    .reset_index()
)


first_position_rows = (
    request_contract_table.query(
        "within_pair_position == 1"
    )
)

request_contract_audit_summary = pd.Series(
    {
        "paired_input_record_count": len(
            structured_records
        ),
        "structured_request_count": int(
            (
                request_contract_table[
                    "condition"
                ]
                == "structured"
            ).sum()
        ),
        "deterministic_text_request_count": int(
            (
                request_contract_table[
                    "condition"
                ]
                == "deterministic_text"
            ).sum()
        ),
        "structured_first_pair_count": int(
            (
                first_position_rows[
                    "order_group"
                ]
                == "structured_first"
            ).sum()
        ),
        "deterministic_text_first_pair_count": int(
            (
                first_position_rows[
                    "order_group"
                ]
                == "deterministic_text_first"
            ).sum()
        ),
        "canonical_payload_hashes_matched": int(
            request_contract_table[
                "canonical_payload_sha256_matches"
            ].sum()
        ),
        "prompt_hashes_rebuilt_and_matched": int(
            request_contract_table[
                "prompt_sha256_matches"
            ].sum()
        ),
        "execution_contract_hashes_rebuilt_and_matched": int(
            request_contract_table[
                "execution_contract_sha256_matches"
            ].sum()
        ),
        "sample_ids_absent_from_all_prompts": bool(
            request_contract_table[
                "sample_id_absent_from_prompt"
            ].all()
        ),
        "safety_contracts_matched": int(
            request_contract_table[
                "safety_contract_matches"
            ].sum()
        ),
        "ground_truth_loaded": False,
        "network_request_made": False,
        "validation_artifact_written": False,
    },
    name="value",
)

display(
    request_contract_condition_summary
)

request_contract_audit_summary

,condition,request_count,unique_sample_count,payload_hash_matches,prompt_hash_matches,execution_contract_hash_matches,safety_contract_matches
0,deterministic_text,200,200,200,200,200,200
1,structured,200,200,200,200,200,200


paired_input_record_count                          200
structured_request_count                           200
deterministic_text_request_count                   200
structured_first_pair_count                        100
deterministic_text_first_pair_count                100
canonical_payload_hashes_matched                   400
prompt_hashes_rebuilt_and_matched                  400
execution_contract_hashes_rebuilt_and_matched      400
sample_ids_absent_from_all_prompts                True
safety_contracts_matched                           400
ground_truth_loaded                              False
network_request_made                             False
validation_artifact_written                      False
Name: value, dtype: object

In [4]:
def find_response_object_schema(schema_document: dict) -> dict:
    """Locate the unique nested schema for the visible response object."""
    matches = []

    def visit(value):
        if isinstance(value, dict):
            properties = value.get("properties")
            if (
                isinstance(properties, dict)
                and "predicted_label" in properties
                and "cited_features" in properties
            ):
                matches.append(value)

            for nested_value in value.values():
                visit(nested_value)

        elif isinstance(value, list):
            for nested_value in value:
                visit(nested_value)

    visit(schema_document)

    if len(matches) != 1:
        raise ValueError(
            "Expected exactly one response-object schema, found "
            f"{len(matches)}."
        )

    return matches[0]


RESPONSE_OBJECT_SCHEMA = find_response_object_schema(
    llm_output_schema
)

Draft202012Validator.check_schema(RESPONSE_OBJECT_SCHEMA)
LLM_OUTPUT_VALIDATOR = Draft202012Validator(
    RESPONSE_OBJECT_SCHEMA
)

PREDICTED_LABEL_SCHEMA = RESPONSE_OBJECT_SCHEMA[
    "properties"
]["predicted_label"]

CITED_FEATURES_SCHEMA = RESPONSE_OBJECT_SCHEMA[
    "properties"
]["cited_features"]

ALLOWED_PREDICTED_LABELS = set(
    PREDICTED_LABEL_SCHEMA.get("enum", ["Benign", "DoS"])
)

MINIMUM_CITATION_COUNT = int(
    CITED_FEATURES_SCHEMA.get("minItems", 5)
)

MAXIMUM_CITATION_COUNT = int(
    CITED_FEATURES_SCHEMA.get("maxItems", 5)
)

assert ALLOWED_PREDICTED_LABELS == {"Benign", "DoS"}
assert MINIMUM_CITATION_COUNT == 5
assert MAXIMUM_CITATION_COUNT == 5


def parse_visible_json_object(visible_response: str) -> dict:
    """Parse the visible response as one bare JSON object."""
    if not isinstance(visible_response, str):
        raise TypeError("visible_response must be a string.")

    try:
        parsed_response = json.loads(visible_response)
    except json.JSONDecodeError as exc:
        raise ValueError(
            f"Visible response is not valid JSON: {exc}"
        ) from exc

    if not isinstance(parsed_response, dict):
        raise ValueError(
            "Visible response must be one JSON object."
        )

    return parsed_response


def validate_schema_structure(
    parsed_response: dict,
) -> tuple[bool, list[str]]:
    """Validate a response against the locked response-object schema."""
    errors = sorted(
        LLM_OUTPUT_VALIDATOR.iter_errors(parsed_response),
        key=lambda error: list(error.absolute_path),
    )

    formatted_errors = []

    for error in errors:
        error_path = ".".join(
            str(item) for item in error.absolute_path
        )
        location = error_path if error_path else "<root>"
        formatted_errors.append(
            f"{location}: {error.message}"
        )

    return len(formatted_errors) == 0, formatted_errors


def extract_grounding_reference(
    structured_model_input: str,
) -> dict:
    """Extract the common 46-feature grounding reference."""
    parsed_input = json.loads(structured_model_input)

    if not isinstance(parsed_input, dict):
        raise ValueError(
            "Structured model_input is not an object."
        )

    features = parsed_input.get("features")

    if not isinstance(features, list):
        raise ValueError(
            "Structured model_input has no features list."
        )

    grounding_reference = {}

    for position, feature in enumerate(features):
        if not isinstance(feature, dict):
            raise ValueError(
                f"Feature {position} is not an object."
            )

        feature_name = feature.get("name")

        if not isinstance(feature_name, str) or not feature_name:
            raise ValueError(
                f"Feature {position} has no valid name."
            )

        if "value" not in feature:
            raise ValueError(
                f"Feature {feature_name!r} has no value."
            )

        if feature_name in grounding_reference:
            raise ValueError(
                f"Duplicate feature name: {feature_name}"
            )

        grounding_reference[feature_name] = feature["value"]

    if len(grounding_reference) != 46:
        raise ValueError(
            "Expected 46 distinct grounding features, found "
            f"{len(grounding_reference)}."
        )

    return grounding_reference


def validate_feature_grounding(
    parsed_response: dict,
    grounding_reference: dict,
) -> dict:
    """Validate label, citation names and exact observed values."""
    predicted_label = parsed_response.get("predicted_label")
    cited_features = parsed_response.get("cited_features")

    predicted_label_valid = (
        isinstance(predicted_label, str)
        and predicted_label in ALLOWED_PREDICTED_LABELS
    )

    citation_count_valid = (
        isinstance(cited_features, list)
        and MINIMUM_CITATION_COUNT
        <= len(cited_features)
        <= MAXIMUM_CITATION_COUNT
    )

    cited_feature_names = []
    unsupported_feature_names = []
    observed_value_mismatches = []

    if isinstance(cited_features, list):
        for position, citation in enumerate(cited_features):
            if not isinstance(citation, dict):
                unsupported_feature_names.append(
                    f"<malformed citation {position}>"
                )
                continue

            feature_name = citation.get("feature_name")
            observed_value = citation.get("observed_value")

            if not isinstance(feature_name, str):
                unsupported_feature_names.append(
                    f"<missing feature name {position}>"
                )
                continue

            cited_feature_names.append(feature_name)

            if feature_name not in grounding_reference:
                unsupported_feature_names.append(feature_name)
                continue

            expected_value = str(
                grounding_reference[feature_name]
            )

            if (
                not isinstance(observed_value, str)
                or observed_value.strip() != expected_value
            ):
                observed_value_mismatches.append(
                    {
                        "feature_name": feature_name,
                        "expected": expected_value,
                        "observed": observed_value,
                    }
                )

    supported_feature_names_valid = (
        citation_count_valid
        and not unsupported_feature_names
        and len(cited_feature_names) == len(cited_features)
    )

    distinct_feature_names_valid = (
        citation_count_valid
        and len(cited_feature_names)
        == len(set(cited_feature_names))
    )

    observed_values_match = (
        citation_count_valid
        and supported_feature_names_valid
        and not observed_value_mismatches
    )

    grounding_valid = all(
        [
            citation_count_valid,
            supported_feature_names_valid,
            distinct_feature_names_valid,
            observed_values_match,
        ]
    )

    return {
        "predicted_label_valid": predicted_label_valid,
        "citation_count_valid": citation_count_valid,
        "supported_feature_names_valid": (
            supported_feature_names_valid
        ),
        "distinct_feature_names_valid": (
            distinct_feature_names_valid
        ),
        "observed_values_match": observed_values_match,
        "grounding_valid": grounding_valid,
        "cited_feature_names": cited_feature_names,
        "unsupported_feature_names": sorted(
            set(unsupported_feature_names)
        ),
        "observed_value_mismatches": (
            observed_value_mismatches
        ),
    }


VALIDATION_BOOLEAN_FIELDS = [
    "visible_json_parsed",
    "schema_structure_valid",
    "predicted_label_valid",
    "citation_count_valid",
    "supported_feature_names_valid",
    "distinct_feature_names_valid",
    "observed_values_match",
    "grounding_valid",
    "overall_response_valid",
]


def validate_model_response(
    visible_response: str,
    grounding_reference: dict,
) -> dict:
    """Independently apply all locked response-validity checks."""
    result = {
        field_name: False
        for field_name in VALIDATION_BOOLEAN_FIELDS
    }

    result.update(
        {
            "parsed_response": None,
            "schema_validation_errors": [],
            "grounding_details": None,
            "validation_error_type": None,
            "validation_error_message": None,
        }
    )

    try:
        parsed_response = parse_visible_json_object(
            visible_response
        )
        result["visible_json_parsed"] = True
        result["parsed_response"] = parsed_response

        schema_valid, schema_errors = (
            validate_schema_structure(parsed_response)
        )
        result["schema_structure_valid"] = schema_valid
        result["schema_validation_errors"] = schema_errors

        if schema_valid:
            grounding_details = validate_feature_grounding(
                parsed_response,
                grounding_reference,
            )
            result["grounding_details"] = grounding_details

            for field_name in [
                "predicted_label_valid",
                "citation_count_valid",
                "supported_feature_names_valid",
                "distinct_feature_names_valid",
                "observed_values_match",
                "grounding_valid",
            ]:
                result[field_name] = grounding_details[
                    field_name
                ]

        result["overall_response_valid"] = all(
            [
                result["visible_json_parsed"],
                result["schema_structure_valid"],
                result["predicted_label_valid"],
                result["grounding_valid"],
            ]
        )

    except Exception as exc:
        result["validation_error_type"] = type(exc).__name__
        result["validation_error_message"] = str(exc)

    return result


def determine_exclusion_reason(
    record: dict,
    validation: dict,
) -> str | None:
    """Assign one deterministic primary exclusion reason."""
    if validation["overall_response_valid"]:
        return None

    if record["error_type"] == "TimeoutExpired":
        return "timeout"

    if record["error_type"] is not None:
        return f"execution_error:{record['error_type']}"

    if not record["request_completed"]:
        return "request_not_completed"

    if not validation["visible_json_parsed"]:
        return "visible_json_not_parsed"

    if not validation["schema_structure_valid"]:
        return "schema_invalid"

    if not validation["predicted_label_valid"]:
        return "predicted_label_invalid"

    if not validation["citation_count_valid"]:
        return "citation_count_invalid"

    if not validation["supported_feature_names_valid"]:
        return "unsupported_feature_name"

    if not validation["distinct_feature_names_valid"]:
        return "duplicate_feature_name"

    if not validation["observed_values_match"]:
        return "observed_value_mismatch"

    return "grounding_invalid"


response_audit_rows = []

for result_record in batch_results:
    record_index = result_record["record_index"]

    grounding_reference = extract_grounding_reference(
        structured_records[record_index]["model_input"]
    )

    visible_response = result_record["visible_response"]

    assert isinstance(visible_response, str)
    assert (
        result_record["visible_response_character_count"]
        == len(visible_response)
    )

    validation = validate_model_response(
        visible_response,
        grounding_reference,
    )

    for field_name in VALIDATION_BOOLEAN_FIELDS:
        assert (
            result_record[field_name]
            == validation[field_name]
        )

    parsed_response = validation["parsed_response"]

    predicted_label = (
        parsed_response.get("predicted_label")
        if validation["overall_response_valid"]
        else None
    )

    cited_features = (
        parsed_response.get("cited_features")
        if validation["overall_response_valid"]
        else None
    )

    response_audit_rows.append(
        {
            "request_plan_position": result_record[
                "request_plan_position"
            ],
            "record_index": record_index,
            "sample_id": result_record["sample_id"],
            "condition": result_record["condition"],
            "order_group": result_record["order_group"],
            "within_pair_position": result_record[
                "within_pair_position"
            ],
            "request_completed": result_record[
                "request_completed"
            ],
            "return_code": result_record["return_code"],
            "error_type": result_record["error_type"],
            "budget_exhausted": result_record[
                "budget_exhausted"
            ],
            **{
                field_name: validation[field_name]
                for field_name in VALIDATION_BOOLEAN_FIELDS
            },
            "response_valid": validation[
                "overall_response_valid"
            ],
            "analysis_eligible": validation[
                "overall_response_valid"
            ],
            "exclusion_reason": determine_exclusion_reason(
                result_record,
                validation,
            ),
            "predicted_label": predicted_label,
            "cited_features": cited_features,
            "schema_validation_error_count": len(
                validation["schema_validation_errors"]
            ),
            "unsupported_feature_name_count": (
                len(
                    validation["grounding_details"][
                        "unsupported_feature_names"
                    ]
                )
                if validation["grounding_details"] is not None
                else None
            ),
            "observed_value_mismatch_count": (
                len(
                    validation["grounding_details"][
                        "observed_value_mismatches"
                    ]
                )
                if validation["grounding_details"] is not None
                else None
            ),
        }
    )


response_audit_table = pd.DataFrame(response_audit_rows)

validity_by_sample = (
    response_audit_table
    .pivot(
        index="sample_id",
        columns="condition",
        values="analysis_eligible",
    )
)

assert validity_by_sample.shape == (200, 2)
assert validity_by_sample.notna().all(axis=None)

paired_eligible_sample_ids = set(
    validity_by_sample.index[
        validity_by_sample.all(axis=1)
    ]
)

response_audit_table["paired_analysis_eligible"] = (
    response_audit_table["sample_id"].isin(
        paired_eligible_sample_ids
    )
)


condition_validation_summary = (
    response_audit_table
    .groupby("condition", sort=True)
    .agg(
        planned_requests=("sample_id", "size"),
        completed_requests=("request_completed", "sum"),
        parsed_responses=("visible_json_parsed", "sum"),
        schema_valid_responses=(
            "schema_structure_valid",
            "sum",
        ),
        grounding_valid_responses=("grounding_valid", "sum"),
        analysis_eligible_responses=(
            "analysis_eligible",
            "sum",
        ),
        timeout_count=(
            "exclusion_reason",
            lambda values: int(
                (values == "timeout").sum()
            ),
        ),
    )
    .reset_index()
)

exclusion_summary = (
    response_audit_table.loc[
        response_audit_table["analysis_eligible"].eq(False),
        ["condition", "exclusion_reason"],
    ]
    .groupby(
        ["condition", "exclusion_reason"],
        sort=True,
    )
    .size()
    .rename("excluded_record_count")
    .reset_index()
)

pair_valid_counts = (
    validity_by_sample
    .sum(axis=1)
    .value_counts()
)

paired_availability_summary = pd.Series(
    {
        "both_conditions_valid": int(
            pair_valid_counts.get(2, 0)
        ),
        "exactly_one_condition_valid": int(
            pair_valid_counts.get(1, 0)
        ),
        "neither_condition_valid": int(
            pair_valid_counts.get(0, 0)
        ),
    },
    name="sample_count",
)


assert len(response_audit_table) == 400

assert int(
    response_audit_table["analysis_eligible"].sum()
) == batch_manifest["response_status"][
    "overall_response_valid_count"
]

assert len(
    paired_eligible_sample_ids
) == batch_manifest["paired_response_availability"][
    "both_conditions_valid_count"
]

assert paired_availability_summary.to_dict() == {
    "both_conditions_valid": 178,
    "exactly_one_condition_valid": 3,
    "neither_condition_valid": 19,
}

assert int(
    (
        response_audit_table["exclusion_reason"]
        == "timeout"
    ).sum()
) == 40

assert int(
    (
        response_audit_table["exclusion_reason"]
        == "schema_invalid"
    ).sum()
) == 1

assert response_audit_table.loc[
    response_audit_table["analysis_eligible"],
    "exclusion_reason",
].isna().all()

assert response_audit_table.loc[
    ~response_audit_table["analysis_eligible"],
    "predicted_label",
].isna().all()


response_validation_summary = pd.Series(
    {
        "audited_request_count": len(
            response_audit_table
        ),
        "independently_reparsed_count": int(
            response_audit_table[
                "visible_json_parsed"
            ].sum()
        ),
        "schema_valid_count": int(
            response_audit_table[
                "schema_structure_valid"
            ].sum()
        ),
        "grounding_valid_count": int(
            response_audit_table[
                "grounding_valid"
            ].sum()
        ),
        "analysis_eligible_condition_count": int(
            response_audit_table[
                "analysis_eligible"
            ].sum()
        ),
        "excluded_condition_count": int(
            (
                ~response_audit_table[
                    "analysis_eligible"
                ]
            ).sum()
        ),
        "timeout_exclusion_count": int(
            (
                response_audit_table["exclusion_reason"]
                == "timeout"
            ).sum()
        ),
        "schema_exclusion_count": int(
            (
                response_audit_table["exclusion_reason"]
                == "schema_invalid"
            ).sum()
        ),
        "paired_analysis_eligible_sample_count": len(
            paired_eligible_sample_ids
        ),
        "stored_validation_flags_all_reproduced": True,
        "ground_truth_loaded": False,
        "network_request_made": False,
        "validation_artifact_written": False,
    },
    name="value",
)

display(condition_validation_summary)
display(exclusion_summary)
display(paired_availability_summary)

response_validation_summary

,condition,planned_requests,completed_requests,parsed_responses,schema_valid_responses,grounding_valid_responses,analysis_eligible_responses,timeout_count
0,deterministic_text,200,181,181,180,180,180,19
1,structured,200,179,179,179,179,179,21


,condition,exclusion_reason,excluded_record_count
0,deterministic_text,schema_invalid,1
1,deterministic_text,timeout,19
2,structured,timeout,21


both_conditions_valid          178
exactly_one_condition_valid      3
neither_condition_valid         19
Name: sample_count, dtype: int64

audited_request_count                       400
independently_reparsed_count                360
schema_valid_count                          359
grounding_valid_count                       359
analysis_eligible_condition_count           359
excluded_condition_count                     41
timeout_exclusion_count                      40
schema_exclusion_count                        1
paired_analysis_eligible_sample_count       178
stored_validation_flags_all_reproduced     True
ground_truth_loaded                       False
network_request_made                      False
validation_artifact_written               False
Name: value, dtype: object

## 4. Construct the frozen inference dataset in memory

This section constructs the analysis-ready inference dataset without modifying or discarding the original experimental evidence. All 400 condition-level records are retained, including timeouts and the schema-invalid response.

Independent validation outcomes are added as explicit eligibility fields:

- `analysis_eligible` identifies a valid condition-level model response.
- `paired_analysis_eligible` identifies samples for which both representations produced valid responses.
- `exclusion_reason` records why an invalid response is excluded from inferential analysis.
- `predicted_label` and `cited_features` are exposed only for independently validated responses.

The resulting dataset is checked entirely in memory before any artifact is written. Private ground truth is not loaded, and no network request is made.

In [5]:
# Build a one-to-one lookup between each formal request and its
# independently reproduced Notebook 08 validation result.
response_audit_lookup = {
    (row["record_index"], row["condition"]): row
    for row in response_audit_table.to_dict(orient="records")
}

assert len(response_audit_lookup) == 400


frozen_inference_records = []

for source_record in batch_results:
    request_key = (
        source_record["record_index"],
        source_record["condition"],
    )

    if request_key not in response_audit_lookup:
        raise KeyError(
            "No independent validation result was found for "
            f"request key {request_key}."
        )

    audit_record = response_audit_lookup[request_key]

    analysis_eligible = bool(
        audit_record["analysis_eligible"]
    )

    paired_analysis_eligible = bool(
        audit_record["paired_analysis_eligible"]
    )

    exclusion_reason = audit_record["exclusion_reason"]

    if pd.isna(exclusion_reason):
        exclusion_reason = None

    # Copy the complete formal-batch record so execution failures and
    # the original visible response remain preserved as evidence.
    frozen_record = dict(source_record)

    # Add the independently derived Notebook 08 analysis fields.
    frozen_record.update(
        {
            "independently_revalidated": True,
            "analysis_eligible": analysis_eligible,
            "paired_analysis_eligible": (
                paired_analysis_eligible
            ),
            "exclusion_reason": exclusion_reason,
            "predicted_label": (
                audit_record["predicted_label"]
                if analysis_eligible
                else None
            ),
            "cited_features": (
                audit_record["cited_features"]
                if analysis_eligible
                else None
            ),
            "schema_validation_error_count": int(
                audit_record[
                    "schema_validation_error_count"
                ]
            ),
            "unsupported_feature_name_count": (
                int(
                    audit_record[
                        "unsupported_feature_name_count"
                    ]
                )
                if pd.notna(
                    audit_record[
                        "unsupported_feature_name_count"
                    ]
                )
                else None
            ),
            "observed_value_mismatch_count": (
                int(
                    audit_record[
                        "observed_value_mismatch_count"
                    ]
                )
                if pd.notna(
                    audit_record[
                        "observed_value_mismatch_count"
                    ]
                )
                else None
            ),
        }
    )

    frozen_inference_records.append(frozen_record)


# Confirm that constructing the frozen dataset did not alter any field
# inherited from the original Notebook 07 aggregate.
assert len(frozen_inference_records) == len(batch_results) == 400

for source_record, frozen_record in zip(
    batch_results,
    frozen_inference_records,
):
    for field_name, source_value in source_record.items():
        assert frozen_record[field_name] == source_value


# Confirm complete and unique request coverage.
frozen_request_keys = [
    (
        record["record_index"],
        record["sample_id"],
        record["condition"],
    )
    for record in frozen_inference_records
]

assert len(frozen_request_keys) == 400
assert len(set(frozen_request_keys)) == 400

frozen_sample_condition_counts = pd.Series(
    [
        record["sample_id"]
        for record in frozen_inference_records
    ]
).value_counts()

assert len(frozen_sample_condition_counts) == 200
assert frozen_sample_condition_counts.eq(2).all()


# Validate the analysis-facing fields. Invalid responses remain in the
# frozen dataset but cannot contribute predictions or citations.
for record in frozen_inference_records:
    if record["analysis_eligible"]:
        assert record["exclusion_reason"] is None
        assert record["predicted_label"] in {
            "Benign",
            "DoS",
        }
        assert isinstance(record["cited_features"], list)
        assert len(record["cited_features"]) == 5

    else:
        assert record["exclusion_reason"] is not None
        assert record["predicted_label"] is None
        assert record["cited_features"] is None


# Create a tabular view for pre-freeze count and coverage checks.
frozen_inference_table = pd.DataFrame(
    frozen_inference_records
)

assert len(frozen_inference_table) == 400
assert int(
    frozen_inference_table["analysis_eligible"].sum()
) == 359

assert int(
    (~frozen_inference_table["analysis_eligible"]).sum()
) == 41

# Each of the 178 complete pairs contributes two condition-level rows.
assert int(
    frozen_inference_table[
        "paired_analysis_eligible"
    ].sum()
) == 356

assert (
    frozen_inference_table.loc[
        frozen_inference_table["paired_analysis_eligible"],
        "sample_id",
    ].nunique()
    == 178
)


# Verify that all failure evidence remains present rather than being
# removed or silently converted into classification outcomes.
preserved_exclusion_counts = (
    frozen_inference_table.loc[
        ~frozen_inference_table["analysis_eligible"],
        ["condition", "exclusion_reason"],
    ]
    .groupby(
        ["condition", "exclusion_reason"],
        sort=True,
    )
    .size()
    .rename("preserved_record_count")
    .reset_index()
)

assert int(
    (
        frozen_inference_table["exclusion_reason"]
        == "timeout"
    ).sum()
) == 40

assert int(
    (
        frozen_inference_table["exclusion_reason"]
        == "schema_invalid"
    ).sum()
) == 1


# Summarise the exact analysis coverage that will be carried forward.
pre_freeze_condition_summary = (
    frozen_inference_table
    .groupby("condition", sort=True)
    .agg(
        retained_record_count=("sample_id", "size"),
        analysis_eligible_count=(
            "analysis_eligible",
            "sum",
        ),
        excluded_count=(
            "analysis_eligible",
            lambda values: int((~values).sum()),
        ),
        paired_analysis_eligible_count=(
            "paired_analysis_eligible",
            "sum",
        ),
    )
    .reset_index()
)

assert (
    pre_freeze_condition_summary.set_index(
        "condition"
    )["retained_record_count"].to_dict()
    == {
        "deterministic_text": 200,
        "structured": 200,
    }
)

assert (
    pre_freeze_condition_summary.set_index(
        "condition"
    )["analysis_eligible_count"].to_dict()
    == {
        "deterministic_text": 180,
        "structured": 179,
    }
)

assert (
    pre_freeze_condition_summary.set_index(
        "condition"
    )["paired_analysis_eligible_count"].to_dict()
    == {
        "deterministic_text": 178,
        "structured": 178,
    }
)


# Serialize deterministically in memory. The next section will write
# these exact bytes atomically after validating the final manifest.
frozen_jsonl_text = "".join(
    json.dumps(
        record,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
        allow_nan=False,
    )
    + "\n"
    for record in frozen_inference_records
)

frozen_jsonl_bytes = frozen_jsonl_text.encode("utf-8")

frozen_inference_sha256 = hashlib.sha256(
    frozen_jsonl_bytes
).hexdigest()

pre_freeze_summary = pd.Series(
    {
        "retained_condition_record_count": len(
            frozen_inference_records
        ),
        "retained_sample_count": (
            frozen_inference_table["sample_id"].nunique()
        ),
        "analysis_eligible_condition_count": int(
            frozen_inference_table[
                "analysis_eligible"
            ].sum()
        ),
        "excluded_condition_count": int(
            (
                ~frozen_inference_table[
                    "analysis_eligible"
                ]
            ).sum()
        ),
        "paired_analysis_eligible_sample_count": (
            frozen_inference_table.loc[
                frozen_inference_table[
                    "paired_analysis_eligible"
                ],
                "sample_id",
            ].nunique()
        ),
        "timeout_evidence_preserved": int(
            (
                frozen_inference_table["exclusion_reason"]
                == "timeout"
            ).sum()
        ),
        "schema_invalid_evidence_preserved": int(
            (
                frozen_inference_table["exclusion_reason"]
                == "schema_invalid"
            ).sum()
        ),
        "frozen_jsonl_size_bytes": len(
            frozen_jsonl_bytes
        ),
        "frozen_jsonl_sha256": (
            frozen_inference_sha256
        ),
        "ground_truth_loaded": False,
        "network_request_made": False,
        "validation_artifact_written": False,
    },
    name="value",
)

display(pre_freeze_condition_summary)
display(preserved_exclusion_counts)

pre_freeze_summary

,condition,retained_record_count,analysis_eligible_count,excluded_count,paired_analysis_eligible_count
0,deterministic_text,200,180,20,178
1,structured,200,179,21,178


,condition,exclusion_reason,preserved_record_count
0,deterministic_text,schema_invalid,1
1,deterministic_text,timeout,19
2,structured,timeout,21


retained_condition_record_count                                                        400
retained_sample_count                                                                  200
analysis_eligible_condition_count                                                      359
excluded_condition_count                                                                41
paired_analysis_eligible_sample_count                                                  178
timeout_evidence_preserved                                                              40
schema_invalid_evidence_preserved                                                        1
frozen_jsonl_size_bytes                                                            1978434
frozen_jsonl_sha256                      773761e420b06b6b19057c930da57c33bbe29d063d2a04...
ground_truth_loaded                                                                  False
network_request_made                                                                 False

## 5. Atomically freeze the independently validated inference dataset

The complete 400-record inference dataset is now frozen as the authoritative input to downstream analysis. The frozen dataset retains successful responses, timeout evidence, and schema-invalid evidence without modification. Condition-level and paired-analysis eligibility are recorded explicitly, while invalid responses retain null analysis-facing predictions.

The JSONL artifact is serialized deterministically and protected by a SHA-256 digest. A separate validation manifest records source-artifact hashes, validation rules, response coverage, exclusion counts, and the exact frozen-dataset hash.

Both files are written atomically. Existing identical files are verified and retained, while divergent existing files are never overwritten. No private ground truth is loaded, and no network request is made.

In [6]:
import os
import tempfile
from datetime import datetime, timezone


def build_artifact_descriptor(path: Path) -> dict:
    """Describe one source artifact using its exact on-disk bytes."""
    return {
        "path": project_relative_path(path),
        "size_bytes": path.stat().st_size,
        "sha256": sha256_file(path),
    }


def atomic_write_new_or_identical(
    destination_path: Path,
    content_bytes: bytes,
) -> str:
    """
    Atomically create an artifact without overwriting divergent evidence.

    If an identical file already exists, it is verified and retained.
    If a different file exists at the same path, execution stops.
    """
    destination_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    if destination_path.exists():
        existing_bytes = destination_path.read_bytes()

        if existing_bytes != content_bytes:
            raise FileExistsError(
                "Refusing to overwrite a divergent existing artifact: "
                f"{destination_path}"
            )

        return "existing_verified"

    file_descriptor, temporary_name = tempfile.mkstemp(
        prefix=f".{destination_path.name}.",
        suffix=".tmp",
        dir=destination_path.parent,
    )

    temporary_path = Path(temporary_name)

    try:
        with os.fdopen(file_descriptor, "wb") as temporary_file:
            temporary_file.write(content_bytes)
            temporary_file.flush()
            os.fsync(temporary_file.fileno())

        # Recheck the destination immediately before replacement so a
        # concurrently created divergent file cannot be overwritten.
        if destination_path.exists():
            existing_bytes = destination_path.read_bytes()

            if existing_bytes != content_bytes:
                raise FileExistsError(
                    "Refusing to overwrite a divergent artifact that "
                    f"appeared during the write: {destination_path}"
                )

            temporary_path.unlink()
            return "existing_verified"

        os.replace(
            temporary_path,
            destination_path,
        )

    except Exception:
        if temporary_path.exists():
            temporary_path.unlink()
        raise

    return "created"


# Reconfirm that the exact in-memory bytes still represent all 400
# records before preparing the permanent validation manifest.
assert len(frozen_inference_records) == 400

assert hashlib.sha256(
    frozen_jsonl_bytes
).hexdigest() == frozen_inference_sha256

assert frozen_inference_sha256 == stable_json_sha256(
    [
        json.loads(line)
        for line in frozen_jsonl_text.splitlines()
        if line.strip()
    ]
) or frozen_inference_sha256 == hashlib.sha256(
    frozen_jsonl_bytes
).hexdigest()


# If an identical manifest was already created during an earlier run,
# reuse its original completion timestamp. This makes notebook restart
# behaviour verifiable without changing frozen provenance.
existing_validation_completed_at_utc = None

if VALIDATION_MANIFEST_PATH.exists():
    existing_validation_manifest = json.loads(
        VALIDATION_MANIFEST_PATH.read_text(
            encoding="utf-8"
        )
    )

    existing_validation_completed_at_utc = (
        existing_validation_manifest.get(
            "validation_completed_at_utc"
        )
    )

validation_completed_at_utc = (
    existing_validation_completed_at_utc
    if existing_validation_completed_at_utc is not None
    else datetime.now(timezone.utc).isoformat()
)


# Record the exact seven locked inputs independently verified earlier
# in Notebook 08.
source_artifacts = {
    "structured_representation": (
        build_artifact_descriptor(STRUCTURED_PATH)
    ),
    "deterministic_text_representation": (
        build_artifact_descriptor(TEXT_PATH)
    ),
    "equivalence_validation": (
        build_artifact_descriptor(EQUIVALENCE_PATH)
    ),
    "representation_manifest": (
        build_artifact_descriptor(
            REPRESENTATION_MANIFEST_PATH
        )
    ),
    "llm_protocol": (
        build_artifact_descriptor(PROTOCOL_PATH)
    ),
    "llm_output_schema": (
        build_artifact_descriptor(OUTPUT_SCHEMA_PATH)
    ),
    "formal_batch_results": (
        build_artifact_descriptor(
            BATCH_RESULTS_JSONL_PATH
        )
    ),
    "formal_batch_manifest": (
        build_artifact_descriptor(BATCH_MANIFEST_PATH)
    ),
}


validation_manifest = {
    "artifact_type": (
        "independently_validated_frozen_inference_dataset"
    ),
    "notebook": "08_llm_response_validation.ipynb",
    "feature_set_id": "primary_46",
    "validation_completed_at_utc": (
        validation_completed_at_utc
    ),
    "research_boundary": {
        "private_ground_truth_loaded": False,
        "network_request_made": False,
        "model_request_retried": False,
        "model_response_repaired": False,
        "failed_evidence_deleted": False,
    },
    "validation_contract": {
        "response_schema": "locked_llm_output_schema",
        "allowed_predicted_labels": [
            "Benign",
            "DoS",
        ],
        "required_citation_count": 5,
        "citation_feature_names_must_be_supplied": True,
        "citation_feature_names_must_be_distinct": True,
        "observed_values_must_match_exactly": True,
        "condition_analysis_requires_valid_response": True,
        "paired_analysis_requires_both_conditions_valid": True,
        "timeouts_are_not_classification_predictions": True,
    },
    "source_artifacts": source_artifacts,
    "source_integrity": {
        "locked_artifact_count": 7,
        "locked_artifact_hashes_matched": 7,
        "formal_batch_record_count": 400,
        "formal_checkpoint_file_count": 400,
        "formal_checkpoint_hashes_matched": 400,
        "aggregate_payloads_matched_checkpoints": 400,
        "ordered_request_positions_verified": True,
        "unique_request_key_count": 400,
        "paired_sample_count": 200,
        "prompt_hashes_rebuilt_and_matched": 400,
        "execution_contract_hashes_rebuilt_and_matched": 400,
        "sample_ids_absent_from_all_prompts": True,
        "stored_validation_flags_all_reproduced": True,
    },
    "response_coverage": {
        "retained_condition_record_count": 400,
        "retained_sample_count": 200,
        "structured_planned_count": 200,
        "deterministic_text_planned_count": 200,
        "structured_analysis_eligible_count": 179,
        "deterministic_text_analysis_eligible_count": 180,
        "analysis_eligible_condition_count": 359,
        "excluded_condition_count": 41,
        "both_conditions_valid_sample_count": 178,
        "exactly_one_condition_valid_sample_count": 3,
        "neither_condition_valid_sample_count": 19,
        "paired_analysis_eligible_sample_count": 178,
    },
    "exclusion_evidence": {
        "timeout_count": 40,
        "schema_invalid_count": 1,
        "structured_timeout_count": 21,
        "deterministic_text_timeout_count": 19,
        "deterministic_text_schema_invalid_count": 1,
        "automatic_retry_used": False,
        "invalid_predictions_exposed_for_analysis": False,
    },
    "frozen_inference_dataset": {
        "path": project_relative_path(
            FROZEN_INFERENCE_PATH
        ),
        "serialization": (
            "utf-8 JSON Lines; sorted keys; compact separators"
        ),
        "record_count": 400,
        "sample_count": 200,
        "size_bytes": len(frozen_jsonl_bytes),
        "sha256": frozen_inference_sha256,
        "contains_all_execution_evidence": True,
        "contains_analysis_eligibility_fields": True,
    },
}


validation_manifest_text = (
    json.dumps(
        validation_manifest,
        ensure_ascii=False,
        sort_keys=True,
        indent=2,
        allow_nan=False,
    )
    + "\n"
)

validation_manifest_bytes = (
    validation_manifest_text.encode("utf-8")
)

validation_manifest_sha256 = hashlib.sha256(
    validation_manifest_bytes
).hexdigest()


# Write the frozen JSONL first and its manifest second. If execution is
# interrupted between writes, rerunning this cell verifies the existing
# JSONL before creating the missing manifest.
frozen_inference_action = (
    atomic_write_new_or_identical(
        FROZEN_INFERENCE_PATH,
        frozen_jsonl_bytes,
    )
)

validation_manifest_action = (
    atomic_write_new_or_identical(
        VALIDATION_MANIFEST_PATH,
        validation_manifest_bytes,
    )
)


# Independently reopen both artifacts and verify their exact content.
written_frozen_records = read_jsonl(
    FROZEN_INFERENCE_PATH
)

written_validation_manifest = json.loads(
    VALIDATION_MANIFEST_PATH.read_text(
        encoding="utf-8"
    )
)

assert written_frozen_records == frozen_inference_records
assert written_validation_manifest == validation_manifest

assert len(written_frozen_records) == 400

assert sha256_file(
    FROZEN_INFERENCE_PATH
) == frozen_inference_sha256

assert sha256_file(
    VALIDATION_MANIFEST_PATH
) == validation_manifest_sha256

assert (
    written_validation_manifest[
        "frozen_inference_dataset"
    ]["sha256"]
    == sha256_file(FROZEN_INFERENCE_PATH)
)

assert (
    written_validation_manifest[
        "frozen_inference_dataset"
    ]["size_bytes"]
    == FROZEN_INFERENCE_PATH.stat().st_size
)

assert (
    written_validation_manifest[
        "frozen_inference_dataset"
    ]["record_count"]
    == len(written_frozen_records)
)


# Confirm that no temporary files remain after the atomic writes.
remaining_temporary_files = sorted(
    path
    for path in VALIDATION_RESULTS_DIR.iterdir()
    if path.is_file()
    and path.name.endswith(".tmp")
)

assert remaining_temporary_files == []


freeze_summary = pd.Series(
    {
        "frozen_inference_action": (
            frozen_inference_action
        ),
        "validation_manifest_action": (
            validation_manifest_action
        ),
        "frozen_record_count": len(
            written_frozen_records
        ),
        "frozen_sample_count": len(
            {
                record["sample_id"]
                for record in written_frozen_records
            }
        ),
        "analysis_eligible_condition_count": sum(
            record["analysis_eligible"]
            for record in written_frozen_records
        ),
        "paired_analysis_eligible_sample_count": len(
            {
                record["sample_id"]
                for record in written_frozen_records
                if record["paired_analysis_eligible"]
            }
        ),
        "excluded_evidence_retained_count": sum(
            not record["analysis_eligible"]
            for record in written_frozen_records
        ),
        "frozen_inference_sha256": (
            frozen_inference_sha256
        ),
        "validation_manifest_sha256": (
            validation_manifest_sha256
        ),
        "temporary_file_count": len(
            remaining_temporary_files
        ),
        "divergent_existing_artifact_overwritten": False,
        "ground_truth_loaded": False,
        "network_request_made": False,
        "validation_artifact_written": True,
    },
    name="value",
)

freeze_summary

frozen_inference_action                                                              created
validation_manifest_action                                                           created
frozen_record_count                                                                      400
frozen_sample_count                                                                      200
analysis_eligible_condition_count                                                        359
paired_analysis_eligible_sample_count                                                    178
excluded_evidence_retained_count                                                          41
frozen_inference_sha256                    773761e420b06b6b19057c930da57c33bbe29d063d2a04...
validation_manifest_sha256                 02bf4eb0fa76bf196461ae049740615ebfca1cfdfbfbb2...
temporary_file_count                                                                       0
divergent_existing_artifact_overwritten                               

## 6. Final post-freeze integrity and reproducibility audit

This section reopens the frozen artifacts from disk and audits them independently of the write operation. It recomputes all relevant hashes, revalidates every visible response, verifies condition-level and paired-analysis eligibility, confirms preservation of all failure evidence, and checks that no source record was modified during freezing.

This is the final technical gate before Notebook 08 is declared complete. Private ground truth remains unopened, and no network request is made.

In [7]:
# Reopen the frozen artifacts from disk. These objects are deliberately
# distinct from the in-memory objects used during artifact creation.
disk_frozen_records = read_jsonl(
    FROZEN_INFERENCE_PATH
)

disk_validation_manifest = json.loads(
    VALIDATION_MANIFEST_PATH.read_text(
        encoding="utf-8"
    )
)

disk_frozen_sha256 = sha256_file(
    FROZEN_INFERENCE_PATH
)

disk_validation_manifest_sha256 = sha256_file(
    VALIDATION_MANIFEST_PATH
)


# Verify the frozen dataset against the hash, size and count recorded
# in its validation manifest.
frozen_descriptor = disk_validation_manifest[
    "frozen_inference_dataset"
]

assert disk_frozen_sha256 == frozen_descriptor["sha256"]

assert (
    FROZEN_INFERENCE_PATH.stat().st_size
    == frozen_descriptor["size_bytes"]
)

assert len(disk_frozen_records) == (
    frozen_descriptor["record_count"]
) == 400


# Recompute every source-artifact hash recorded in the manifest.
source_artifact_audit_rows = []

for artifact_name, descriptor in (
    disk_validation_manifest[
        "source_artifacts"
    ].items()
):
    artifact_path = (
        PROJECT_ROOT / descriptor["path"]
    ).resolve()

    assert artifact_path.exists()
    assert artifact_path.is_file()

    observed_size_bytes = artifact_path.stat().st_size
    observed_sha256 = sha256_file(artifact_path)

    size_matched = (
        observed_size_bytes
        == descriptor["size_bytes"]
    )

    sha256_matched = (
        observed_sha256
        == descriptor["sha256"]
    )

    assert size_matched
    assert sha256_matched

    source_artifact_audit_rows.append(
        {
            "artifact_name": artifact_name,
            "path": descriptor["path"],
            "size_matched": size_matched,
            "sha256_matched": sha256_matched,
        }
    )

source_artifact_audit_table = pd.DataFrame(
    source_artifact_audit_rows
)

assert len(source_artifact_audit_table) == 8
assert source_artifact_audit_table[
    "size_matched"
].all()
assert source_artifact_audit_table[
    "sha256_matched"
].all()


# Confirm that every Notebook 07 source field is still present and
# byte-equivalent at the JSON-value level in the frozen dataset.
assert len(batch_results) == len(disk_frozen_records)

for source_record, frozen_record in zip(
    batch_results,
    disk_frozen_records,
):
    assert (
        source_record["request_plan_position"]
        == frozen_record["request_plan_position"]
    )

    assert (
        source_record["record_index"]
        == frozen_record["record_index"]
    )

    assert (
        source_record["sample_id"]
        == frozen_record["sample_id"]
    )

    assert (
        source_record["condition"]
        == frozen_record["condition"]
    )

    for field_name, source_value in source_record.items():
        assert field_name in frozen_record
        assert frozen_record[field_name] == source_value


# Independently rerun the complete visible-response validation after
# reading the frozen dataset from disk.
post_freeze_validation_rows = []

for frozen_record in disk_frozen_records:
    record_index = frozen_record["record_index"]

    grounding_reference = extract_grounding_reference(
        structured_records[record_index]["model_input"]
    )

    repeated_validation = validate_model_response(
        frozen_record["visible_response"],
        grounding_reference,
    )

    for field_name in VALIDATION_BOOLEAN_FIELDS:
        assert (
            frozen_record[field_name]
            == repeated_validation[field_name]
        )

    repeated_analysis_eligible = bool(
        repeated_validation["overall_response_valid"]
    )

    assert (
        frozen_record["analysis_eligible"]
        == repeated_analysis_eligible
    )

    expected_exclusion_reason = determine_exclusion_reason(
        frozen_record,
        repeated_validation,
    )

    assert (
        frozen_record["exclusion_reason"]
        == expected_exclusion_reason
    )

    if repeated_analysis_eligible:
        repeated_parsed_response = (
            repeated_validation["parsed_response"]
        )

        assert frozen_record["predicted_label"] == (
            repeated_parsed_response["predicted_label"]
        )

        assert frozen_record["cited_features"] == (
            repeated_parsed_response["cited_features"]
        )

        assert (
            frozen_record["predicted_label"]
            in ALLOWED_PREDICTED_LABELS
        )

        assert len(
            frozen_record["cited_features"]
        ) == 5

    else:
        assert frozen_record["predicted_label"] is None
        assert frozen_record["cited_features"] is None

    post_freeze_validation_rows.append(
        {
            "request_plan_position": frozen_record[
                "request_plan_position"
            ],
            "record_index": record_index,
            "sample_id": frozen_record["sample_id"],
            "condition": frozen_record["condition"],
            "request_completed": frozen_record[
                "request_completed"
            ],
            "analysis_eligible": (
                repeated_analysis_eligible
            ),
            "paired_analysis_eligible": frozen_record[
                "paired_analysis_eligible"
            ],
            "exclusion_reason": expected_exclusion_reason,
        }
    )


post_freeze_validation_table = pd.DataFrame(
    post_freeze_validation_rows
)

assert len(post_freeze_validation_table) == 400


# Reconstruct paired eligibility solely from the condition-level
# validation results stored in the frozen artifact.
disk_validity_by_sample = (
    post_freeze_validation_table
    .pivot(
        index="sample_id",
        columns="condition",
        values="analysis_eligible",
    )
)

assert disk_validity_by_sample.shape == (200, 2)
assert disk_validity_by_sample.notna().all(axis=None)

expected_paired_eligibility = (
    disk_validity_by_sample.all(axis=1)
)

assert int(
    expected_paired_eligibility.sum()
) == 178


# Every row belonging to a sample must carry the same paired flag, and
# that flag must equal the eligibility reconstructed above.
for sample_id, sample_rows in (
    post_freeze_validation_table.groupby(
        "sample_id",
        sort=False,
    )
):
    stored_pair_flags = set(
        sample_rows[
            "paired_analysis_eligible"
        ].tolist()
    )

    assert len(stored_pair_flags) == 1

    stored_pair_flag = stored_pair_flags.pop()

    assert stored_pair_flag == bool(
        expected_paired_eligibility.loc[sample_id]
    )


# Recompute condition-level coverage and exclusion evidence from the
# disk-loaded frozen dataset.
post_freeze_condition_summary = (
    post_freeze_validation_table
    .groupby("condition", sort=True)
    .agg(
        retained_record_count=("sample_id", "size"),
        completed_request_count=(
            "request_completed",
            "sum",
        ),
        analysis_eligible_count=(
            "analysis_eligible",
            "sum",
        ),
        paired_analysis_eligible_count=(
            "paired_analysis_eligible",
            "sum",
        ),
    )
    .reset_index()
)

post_freeze_exclusion_summary = (
    post_freeze_validation_table.loc[
        ~post_freeze_validation_table[
            "analysis_eligible"
        ],
        ["condition", "exclusion_reason"],
    ]
    .groupby(
        ["condition", "exclusion_reason"],
        sort=True,
    )
    .size()
    .rename("preserved_record_count")
    .reset_index()
)

condition_summary_by_name = (
    post_freeze_condition_summary
    .set_index("condition")
)

assert (
    condition_summary_by_name[
        "retained_record_count"
    ].to_dict()
    == {
        "deterministic_text": 200,
        "structured": 200,
    }
)

assert (
    condition_summary_by_name[
        "completed_request_count"
    ].to_dict()
    == {
        "deterministic_text": 181,
        "structured": 179,
    }
)

assert (
    condition_summary_by_name[
        "analysis_eligible_count"
    ].to_dict()
    == {
        "deterministic_text": 180,
        "structured": 179,
    }
)

assert (
    condition_summary_by_name[
        "paired_analysis_eligible_count"
    ].to_dict()
    == {
        "deterministic_text": 178,
        "structured": 178,
    }
)

assert int(
    post_freeze_validation_table[
        "analysis_eligible"
    ].sum()
) == 359

assert int(
    (
        ~post_freeze_validation_table[
            "analysis_eligible"
        ]
    ).sum()
) == 41

assert int(
    (
        post_freeze_validation_table[
            "exclusion_reason"
        ]
        == "timeout"
    ).sum()
) == 40

assert int(
    (
        post_freeze_validation_table[
            "exclusion_reason"
        ]
        == "schema_invalid"
    ).sum()
) == 1


# Verify the response-coverage counts against the manifest rather than
# relying only on hard-coded expectations.
manifest_coverage = disk_validation_manifest[
    "response_coverage"
]

assert manifest_coverage[
    "retained_condition_record_count"
] == 400

assert manifest_coverage[
    "retained_sample_count"
] == 200

assert manifest_coverage[
    "analysis_eligible_condition_count"
] == 359

assert manifest_coverage[
    "excluded_condition_count"
] == 41

assert manifest_coverage[
    "paired_analysis_eligible_sample_count"
] == 178

assert manifest_coverage[
    "both_conditions_valid_sample_count"
] == 178

assert manifest_coverage[
    "exactly_one_condition_valid_sample_count"
] == 3

assert manifest_coverage[
    "neither_condition_valid_sample_count"
] == 19


# Confirm that the validation directory contains no abandoned atomic
# write files.
remaining_validation_temporary_files = sorted(
    path
    for path in VALIDATION_RESULTS_DIR.rglob("*")
    if path.is_file()
    and path.name.endswith(".tmp")
)

assert remaining_validation_temporary_files == []


display(source_artifact_audit_table)
display(post_freeze_condition_summary)
display(post_freeze_exclusion_summary)

final_post_freeze_audit_summary = pd.Series(
    {
        "source_artifact_count_reverified": len(
            source_artifact_audit_table
        ),
        "source_artifact_hashes_matched": int(
            source_artifact_audit_table[
                "sha256_matched"
            ].sum()
        ),
        "frozen_record_count": len(
            disk_frozen_records
        ),
        "unique_sample_count": (
            post_freeze_validation_table[
                "sample_id"
            ].nunique()
        ),
        "responses_independently_revalidated": len(
            post_freeze_validation_table
        ),
        "analysis_eligible_condition_count": int(
            post_freeze_validation_table[
                "analysis_eligible"
            ].sum()
        ),
        "excluded_evidence_retained_count": int(
            (
                ~post_freeze_validation_table[
                    "analysis_eligible"
                ]
            ).sum()
        ),
        "paired_analysis_eligible_sample_count": int(
            expected_paired_eligibility.sum()
        ),
        "timeout_evidence_preserved": int(
            (
                post_freeze_validation_table[
                    "exclusion_reason"
                ]
                == "timeout"
            ).sum()
        ),
        "schema_invalid_evidence_preserved": int(
            (
                post_freeze_validation_table[
                    "exclusion_reason"
                ]
                == "schema_invalid"
            ).sum()
        ),
        "frozen_inference_sha256": (
            disk_frozen_sha256
        ),
        "validation_manifest_sha256": (
            disk_validation_manifest_sha256
        ),
        "temporary_file_count": len(
            remaining_validation_temporary_files
        ),
        "post_freeze_integrity_gate_passed": True,
        "ground_truth_loaded": False,
        "network_request_made": False,
        "new_artifact_written_by_this_cell": False,
    },
    name="value",
)

final_post_freeze_audit_summary

,artifact_name,path,size_matched,sha256_matched
0,deterministic_text_representation,data/interim/representations/primary_46/determ...,True,True
1,equivalence_validation,data/interim/representations/primary_46/equiva...,True,True
2,formal_batch_manifest,results/inference/primary_46/opencode_batch_ma...,True,True
3,formal_batch_results,results/inference/primary_46/opencode_batch_re...,True,True
4,llm_output_schema,configs/llm_output_schema.json,True,True
5,llm_protocol,configs/llm_protocol_primary_46.json,True,True
6,representation_manifest,data/interim/representations/primary_46/manife...,True,True
7,structured_representation,data/interim/representations/primary_46/struct...,True,True


,condition,retained_record_count,completed_request_count,analysis_eligible_count,paired_analysis_eligible_count
0,deterministic_text,200,181,180,178
1,structured,200,179,179,178


,condition,exclusion_reason,preserved_record_count
0,deterministic_text,schema_invalid,1
1,deterministic_text,timeout,19
2,structured,timeout,21


source_artifact_count_reverified                                                         8
source_artifact_hashes_matched                                                           8
frozen_record_count                                                                    400
unique_sample_count                                                                    200
responses_independently_revalidated                                                    400
analysis_eligible_condition_count                                                      359
excluded_evidence_retained_count                                                        41
paired_analysis_eligible_sample_count                                                  178
timeout_evidence_preserved                                                              40
schema_invalid_evidence_preserved                                                        1
frozen_inference_sha256                  773761e420b06b6b19057c930da57c33bbe29d063d2a04...

## Summary and downstream analysis contract

Notebook 08 independently audited and froze the complete `primary_46` inference dataset produced by Notebook 07. No private ground truth was loaded, no model request was made, and no failed response was retried, repaired, overwritten, or deleted.

### Integrity and protocol verification

The notebook:

- verified the hashes and identities of the locked representations, equivalence evidence, protocol, output schema, formal batch results, formal batch manifest, and all 400 request checkpoints;
- reconstructed and matched all 400 prompt hashes and execution-contract hashes;
- confirmed that the 200 structured requests and 200 deterministic-text requests represent the same 200 canonical input records;
- confirmed the balanced presentation order of 100 structured-first and 100 deterministic-text-first pairs;
- confirmed that sample IDs were absent from every model prompt; and
- reproduced all stored response-validation flags independently.

### Response-validation results

All 400 condition-level records were retained in the frozen dataset.

| Condition | Planned | Completed | Valid and analysis-eligible | Excluded |
|---|---:|---:|---:|---:|
| Structured JSON | 200 | 179 | 179 | 21 |
| Deterministic text | 200 | 181 | 180 | 20 |
| **Total** | **400** | **360** | **359** | **41** |

The 41 excluded condition-level records consist of:

- 40 requests that reached the locked 300-second timeout: 21 structured and 19 deterministic-text requests; and
- one completed deterministic-text response that did not satisfy the locked response schema.

Timeouts are execution outcomes rather than classification predictions. The schema-invalid response is also not treated as a valid prediction. These records remain preserved as experimental evidence, with explicit exclusion reasons and null analysis-facing predictions.

### Paired-response availability

Among the 200 sampled records:

- 178 have valid responses under both representation conditions;
- 3 have a valid response under exactly one condition; and
- 19 have no valid response under either condition.

The frozen dataset therefore supports two distinct analysis populations:

1. **Condition-specific analysis:** 179 valid structured responses and 180 valid deterministic-text responses, with coverage reported explicitly.
2. **Primary paired representation comparison:** 178 complete pairs, because paired statistical procedures require valid outputs under both conditions for the same underlying record.

No assumption is made that timeout-related missingness is random. Execution coverage and failure patterns must therefore be reported separately from classification performance.

### Frozen artifacts

The notebook wrote:

- a deterministic JSONL dataset containing all 400 condition-level records, their original execution evidence, independently reproduced validation results, analysis-eligibility flags, and valid parsed predictions; and
- a validation manifest containing source hashes, validation rules, coverage counts, exclusion counts, and the exact SHA-256 digest of the frozen JSONL dataset.

The exact artifact paths and full SHA-256 digests are reported in the final completion cell and stored in the validation manifest.

### Contract for downstream notebooks

Notebook 09 may load private ground truth for the first time. It must:

- retain all 400 frozen records as the inference audit population;
- report execution coverage separately for each representation;
- use only `analysis_eligible = True` records for condition-specific classification metrics;
- use only the 178 samples marked `paired_analysis_eligible = True` for the primary paired structured-versus-text comparison;
- treat timeouts and invalid responses as missing predictions rather than classification errors; and
- never retry, repair, or replace frozen model responses after labels are available.

Notebook 10 may use the 359 valid condition-level responses for condition-specific attribution summaries and the 178 complete pairs for structured-versus-text attribution agreement. Feature grounding establishes that cited names and values came from the supplied record; it does not establish causal or objectively correct explanations.

The `primary_46` inference dataset is now independently validated, hash-identified, and frozen for downstream evaluation.

In [8]:
# Record the final handoff state for downstream notebooks and reviewers.
assert final_post_freeze_audit_summary[
    "post_freeze_integrity_gate_passed"
]

assert len(disk_frozen_records) == 400
assert int(
    post_freeze_validation_table[
        "analysis_eligible"
    ].sum()
) == 359
assert int(
    expected_paired_eligibility.sum()
) == 178

notebook_08_completion_summary = pd.Series(
    {
        "feature_set_id": "primary_46",
        "frozen_inference_path": project_relative_path(
            FROZEN_INFERENCE_PATH
        ),
        "validation_manifest_path": project_relative_path(
            VALIDATION_MANIFEST_PATH
        ),
        "frozen_inference_sha256": sha256_file(
            FROZEN_INFERENCE_PATH
        ),
        "validation_manifest_sha256": sha256_file(
            VALIDATION_MANIFEST_PATH
        ),
        "retained_condition_record_count": 400,
        "retained_sample_count": 200,
        "structured_analysis_eligible_count": 179,
        "deterministic_text_analysis_eligible_count": 180,
        "analysis_eligible_condition_count": 359,
        "excluded_evidence_retained_count": 41,
        "paired_analysis_eligible_sample_count": 178,
        "ground_truth_loaded": False,
        "network_request_made": False,
        "post_label_response_repair_permitted": False,
        "ready_for_notebook_09": True,
        "notebook_08_complete": True,
    },
    name="value",
)

notebook_08_completion_summary

feature_set_id                                                                       primary_46
frozen_inference_path                         results/validation/primary_46/opencode_frozen_...
validation_manifest_path                      results/validation/primary_46/opencode_validat...
frozen_inference_sha256                       773761e420b06b6b19057c930da57c33bbe29d063d2a04...
validation_manifest_sha256                    02bf4eb0fa76bf196461ae049740615ebfca1cfdfbfbb2...
retained_condition_record_count                                                             400
retained_sample_count                                                                       200
structured_analysis_eligible_count                                                          179
deterministic_text_analysis_eligible_count                                                  180
analysis_eligible_condition_count                                                           359
excluded_evidence_retained_count        